# Hallucination and Factuality Evaluation

## নোটবুক পরিচিতি

একটি বাস্তব (যদিও ছোট) NLI-শৈলীর (entailment / contradiction / neutral)
factuality checker শূন্য থেকে তৈরি করা হয়:

1. কাল্পনিক কোম্পানিগুলোর (প্রতিষ্ঠার বছর, প্রতিষ্ঠাতা, শহর, product) sentence
   template থেকে (source, claim, label) triple-এর একটি বড় synthetic dataset
   generate করা হয়। ENTAILMENT claim source আসলে যা বলে তা পুনর্ব্যক্ত করে;
   CONTRADICTION claim ঠিক একটি fact বদলে দেয়; NEUTRAL claim source কখনো
   উল্লেখই করে না এমন একটি attribute নিয়ে জিজ্ঞেস করে।
2. (source, claim) pair-এর bag-of-words feature-এ একটি ছোট MLP classifier
   প্রশিক্ষণ দিয়ে 3-মুখী label ভবিষ্যদ্বাণী করা হয়।
3. প্রশিক্ষিত checker-টিকে একটি NEW source passage-এর একটি toy generated
   "summary"-এর প্রতিটি বাক্যে প্রয়োগ করা হয় (কয়েকটি বাক্য ইচ্ছাকৃতভাবে
   অসমর্থিত বা সাংঘর্ষিক লেখা), এবং checker-এর hallucination flag-গুলো পরিচিত
   ground truth-এর বিরুদ্ধে precision/recall দিয়ে মাপা হয়।

Runtime: CPU-তে কয়েক সেকেন্ড (ছোট MLP, ~3,600 ছোট bag-of-words example)।

## কীভাবে চালাবেন

মূল file-টি হলো `example.py` — `python example.py` দিয়ে চলে। notebook-এ একই
কোড cell-by-cell চালানো হয়; শেষ cell-টি `main()` কল করে।

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(0)
torch.manual_seed(0)

## Toy world: কাল্পনিক কোম্পানি

প্রতিষ্ঠার তথ্যসহ কাল্পনিক কোম্পানিগুলোর এই তালিকাটি ব্যবহার করে (source, claim,
label) triple-এর কার্যত সীমাহীন সরবরাহ generate করা হয়।

In [ ]:
COMPANIES = ["Acme Corp", "Globex", "Initech", "Umbrella Inc", "Stark Industries",
             "Wayne Enterprises", "Wonka Factory", "Hooli", "Cyberdyne", "Soylent Corp"]
YEARS = list(range(1950, 2021))
FOUNDERS = ["Alice Chen", "Bob Martinez", "Carol Nguyen", "David Kim",
            "Eve Johnson", "Frank Lopez", "Grace Patel", "Henry Wu"]
CITIES = ["Boston", "Seattle", "Austin", "Chicago", "Denver", "Portland", "Atlanta", "Miami"]
PRODUCTS = ["software", "robotics", "chemicals", "electronics", "toys", "vehicles",
            "pharmaceuticals", "food products"]
EMPLOYEE_COUNTS = ["50", "200", "500", "1200", "3000", "10000"]

LABELS = ["entailment", "contradiction", "neutral"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}


def make_source(company, year, founder, city, product):
    return f"{company} was founded in {year} by {founder} in {city}. the company produces {product}."


def random_other(value, options):
    """`value` থেকে ভিন্ন একটি random option বেছে নেওয়ার গ্যারান্টি।"""
    choice = value
    while choice == value:
        choice = random.choice(options)
    return choice


def make_triple():
    """একটি random (source, claim, label) triple sample করা।"""
    company = random.choice(COMPANIES)
    year = random.choice(YEARS)
    founder = random.choice(FOUNDERS)
    city = random.choice(CITIES)
    product = random.choice(PRODUCTS)
    source = make_source(company, year, founder, city, product)

    label = random.choice(LABELS)

    if label == "entailment":
        template = random.choice([
            f"{company} was founded in {year}.",
            f"{founder} founded {company} in {year}.",
            f"{company} is based in {city}.",
            f"{company} makes {product}.",
        ])
        claim = template

    elif label == "contradiction":
        slot = random.choice(["year", "founder", "city", "product"])
        if slot == "year":
            claim = f"{company} was founded in {random_other(year, YEARS)}."
        elif slot == "founder":
            claim = f"{random_other(founder, FOUNDERS)} founded {company} in {year}."
        elif slot == "city":
            claim = f"{company} is based in {random_other(city, CITIES)}."
        else:
            claim = f"{company} makes {random_other(product, PRODUCTS)}."

    else:  # neutral -- source কখনোই সম্বোধন করে না এমন একটি attribute
        claim = random.choice([
            f"{company} has over {random.choice(EMPLOYEE_COUNTS)} employees.",
            f"{company} is publicly traded on the stock exchange.",
            f"{company} won an industry innovation award last year.",
            f"{company}'s chief executive previously worked at a bank.",
        ])

    return source, claim, label


def make_dataset(n):
    return [make_triple() for _ in range(n)]

## Bag-of-words featurization

প্রতিটি (source, claim) pair-কে feature করা হয়: source-এর bag-of-words vector ও
claim-এর bag-of-words vector-কে জোড়া দিয়ে। MLP-কে দুটি bag-এর তুলনা করতে শেখার
দায়িত্ব নিজেকেই নিতে হয় — কাঁচা word count ছাড়া কিছুই hand-engineered নয়।

In [ ]:
def tokenize(text):
    return text.lower().replace(".", "").replace(",", "").replace("'", " ").split()


def build_vocab(triples):
    vocab = set()
    for source, claim, _ in triples:
        vocab.update(tokenize(source))
        vocab.update(tokenize(claim))
    return {word: i for i, word in enumerate(sorted(vocab))}


def bow_vector(text, vocab):
    vec = torch.zeros(len(vocab))
    for tok in tokenize(text):
        if tok in vocab:
            vec[vocab[tok]] += 1.0
    return vec


def featurize(source, claim, vocab):
    """Feature = concat(bag-of-words(source), bag-of-words(claim)) -- MLP-কে
    দুটি bag-এর তুলনা করতে নিজেকেই শিখতে হয়; কাঁচা word count ছাড়া কিছুই
    hand-engineered নয়।"""
    return torch.cat([bow_vector(source, vocab), bow_vector(claim, vocab)])

## Entailment classifier: একটি ছোট MLP

একটি 2-স্তরের MLP (input → ReLU → 3 class), সাথে প্রশিক্ষণ, মূল্যায়ন এবং
একক (source, claim) pair-এর label ভবিষ্যদ্বাণীর ফাংশন।

In [ ]:
class EntailmentMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


def train_classifier(train_triples, vocab, epochs=25, batch_size=64, lr=1e-2):
    X = torch.stack([featurize(s, c, vocab) for s, c, _ in train_triples])
    y = torch.tensor([LABEL_TO_ID[label] for _, _, label in train_triples], dtype=torch.long)

    model = EntailmentMLP(input_dim=X.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    n = X.shape[0]
    for epoch in range(epochs):
        perm = torch.randperm(n)
        total_loss = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            logits = model(X[idx])
            loss = F.cross_entropy(logits, y[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  epoch {epoch + 1:3d}   avg loss = {total_loss / n:.4f}")

    return model


@torch.no_grad()
def evaluate_classifier(model, triples, vocab):
    X = torch.stack([featurize(s, c, vocab) for s, c, _ in triples])
    y = torch.tensor([LABEL_TO_ID[label] for _, _, label in triples], dtype=torch.long)
    preds = model(X).argmax(dim=-1)
    accuracy = (preds == y).float().mean().item()
    return accuracy, preds


@torch.no_grad()
def predict_label(model, source, claim, vocab):
    x = featurize(source, claim, vocab).unsqueeze(0)
    pred_id = model(x).argmax(dim=-1).item()
    return LABELS[pred_id]

## Demo 1: synthetic held-out data-তে checker প্রশিক্ষণ ও মূল্যায়ন

3,600টি synthetic triple-এ classifier-টিকে প্রশিক্ষণ দেওয়া হয়, তারপর 80/20
split-এ train/test accuracy রিপোর্ট করা হয়। প্রশিক্ষিত model ও vocab-কে
পরবর্তী demo-তে ব্যবহারের জন্য রেখে দেওয়া হয়।

In [ ]:
def training_demo():
    print("=" * 78)
    print("1. TRAINING A TOY 3-WAY ENTAILMENT CHECKER ON SYNTHETIC (SOURCE, CLAIM) DATA")
    print("=" * 78)

    all_triples = make_dataset(3600)
    vocab = build_vocab(all_triples)
    random.shuffle(all_triples)
    split = int(0.8 * len(all_triples))
    train_triples, test_triples = all_triples[:split], all_triples[split:]

    print(f"Vocabulary size: {len(vocab)} words")
    print(f"Train examples: {len(train_triples)}   Test examples: {len(test_triples)}")
    counts = {label: sum(1 for _, _, l in all_triples if l == label) for label in LABELS}
    print(f"Label balance (full dataset): {counts}\n")

    print("Training the MLP (bag-of-words features -> hidden 64 -> 3 classes)...")
    model = train_classifier(train_triples, vocab)

    train_acc, _ = evaluate_classifier(model, train_triples, vocab)
    test_acc, _ = evaluate_classifier(model, test_triples, vocab)
    print(f"\nTrain accuracy: {train_acc:.1%}    Test accuracy: {test_acc:.1%}")
    print(f"-> Random guessing among 3 classes would score 33%; {test_acc:.1%} on wordings the")
    print("   model never trained on shows it learned real signal about whether a claim's")
    print("   stated fact matches, conflicts with, or goes unmentioned by the source --")
    print(f"   not just the training sentences themselves. The gap to {train_acc:.0%} train accuracy")
    print("   is expected: bag-of-words features over a fairly small, template-generated")
    print("   world (10 companies, 8 founders, 8 cities) let the MLP memorize part of the")
    print("   training set, which is exactly the honest, well-known trade-off of a simple")
    print("   bag-of-words classifier -- real NLI systems use far richer sentence")
    print("   representations to reduce this gap.")

    return model, vocab


model, vocab = training_demo()

## Demo 2: checker-কে একটি toy generated summary-তে প্রয়োগ

একটি NEW source passage-এর বিরুদ্ধে, injected hallucination-সহ একটি ছোট
summary-এর প্রতিটি বাক্য checker দিয়ে যাচাই করা হয় এবং flag-গুলো পরিচিত
ground truth-এর বিরুদ্ধে precision/recall দিয়ে স্কোর করা হয়।

In [ ]:
def summary_factuality_demo(model, vocab):
    print("\n" + "=" * 78)
    print("2. APPLYING THE CHECKER TO A TOY GENERATED SUMMARY")
    print("=" * 78)

    source = ("novatech was founded in 2004 by maria alvarez in denver. "
              "the company produces electronics.")
    print(f"Source passage:\n  {source!r}\n")

    # প্রতিটি summary বাক্য, সাথে source-এর সাথে তার KNOWN প্রকৃত সম্পর্ক
    # (এই ground truth-টি একজন মানব annotator দেবেন; checker কখনো তা দেখে না)।
    # "entailment" বাক্যগুলো FLAG হওয়া উচিত নয়; বাকি দুটি label হলো
    # hallucination/অসমর্থিত claim এবং FLAG হওয়া উচিত।
    summary_sentences = [
        ("novatech was founded in 2004.",                                  "entailment"),
        ("maria alvarez founded novatech in 2004.",                        "entailment"),
        ("novatech is based in denver.",                                   "entailment"),
        ("novatech makes electronics.",                                    "entailment"),
        ("novatech was founded in 1999.",                                  "contradiction"),   # intrinsic hallucination
        ("james carter founded novatech in 2004.",                         "contradiction"),   # intrinsic hallucination
        ("novatech has over 10000 employees.",                             "neutral"),          # extrinsic / unsupported
        ("novatech won an industry innovation award last year.",           "neutral"),          # extrinsic / unsupported
    ]

    print(f"{'summary sentence':55}{'true label':>14}{'checker verdict':>18}{'flagged?':>10}")
    print("-" * 97)

    true_positive = false_positive = false_negative = true_negative = 0
    for sentence, true_label in summary_sentences:
        predicted = predict_label(model, source, sentence, vocab)
        should_flag = true_label != "entailment"
        is_flagged = predicted != "entailment"

        if should_flag and is_flagged:
            true_positive += 1
        elif (not should_flag) and is_flagged:
            false_positive += 1
        elif should_flag and (not is_flagged):
            false_negative += 1
        else:
            true_negative += 1

        marker = "FLAGGED" if is_flagged else "-"
        print(f"{sentence:55}{true_label:>14}{predicted:>18}{marker:>10}")

    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else float("nan")
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else float("nan")

    print(f"\nConfusion counts on this toy summary's 8 sentences:")
    print(f"  true positives  (hallucination, correctly flagged)     = {true_positive}")
    print(f"  false positives (genuinely supported, wrongly flagged) = {false_positive}")
    print(f"  false negatives (hallucination, MISSED)                = {false_negative}")
    print(f"  true negatives  (genuinely supported, correctly passed) = {true_negative}")
    print(f"\nPrecision = {precision:.2f}   Recall = {recall:.2f}")

    if false_positive == 0 and false_negative == 0:
        print("\n-> The checker flagged every contradiction and every unsupported neutral")
        print("   claim, and passed every genuinely entailed claim, on this toy summary --")
        print(f"   precision={precision:.2f} and recall={recall:.2f}. It is doing exactly what an NLI-based")
        print("   factuality checker is supposed to do: distinguish 'this restates a fact")
        print("   the source actually supports' from 'this contradicts or goes beyond what")
        print("   the source says' using the same 3-way ENTAILMENT/CONTRADICTION/NEUTRAL")
        print("   judgment it was trained on, applied to sentences it has never seen before.")
    else:
        print(f"\n-> On this toy summary the checker made {false_positive} false-positive(s) and")
        print(f"   {false_negative} false-negative(s), giving precision={precision:.2f} and recall={recall:.2f}.")
        print("   Even a checker trained to reasonable held-out accuracy is not perfect --")
        print("   which is exactly why Section 4 of the README stresses that automated")
        print("   factuality checking reduces, but does not eliminate, the need for human")
        print("   review of anything high-stakes.")


summary_factuality_demo(model, vocab)

## সবগুলো demo একসাথে: main()

`main()` দুটি demo একই ক্রমে চালায় — মূল `example.py`-তে এটি
`if __name__ == "__main__":` guard-এর ভেতরে; notebook-এ শেষ cell হিসেবে `main()`
কল করা হয়।

In [ ]:
def main():
    model, vocab = training_demo()
    summary_factuality_demo(model, vocab)


main()